# GP02 — Clean Geospatial Data

Κατανοώ → Εντοπίζω → Διορθώνω → Επαληθεύω → Εξάγω

Δεν μετατρέπουμε προβληματικές τιμές σε αυθαίρετες “σωστές” τιμές. Κρατάμε counts πριν/μετά, δηλώνουμε τους κανόνες και ελέγχουμε ξανά CRS, rows, geometry, bounds και μοναδικά IDs.

## 1. Load — Πριν και μετά

Το dataset έχει σκόπιμα missing values, duplicate ID, μη αριθμητικό elevation και longitude εκτός ορίων.

In [ ]:
import pandas as pd
import geopandas as gpd
df = pd.read_csv("data/gp02/observations_dirty.csv")
print("Αρχικές γραμμές:", len(df))
print(df.dtypes)
print(df.isna().sum())

## 2. Data types και missing values

Το `not_available` γίνεται `NaN`, όχι μηδέν. Αφαιρούμε μόνο rows χωρίς essential fields για Point geometry· τα optional missing πρέπει να παραμένουν ορατά και τεκμηριωμένα.

In [ ]:
for column in ["longitude", "latitude", "elevation_m"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")
print(df.isna().sum())
clean = df.dropna(subset=["observation_id", "longitude", "latitude"]).copy()
print("Μετά το missing-value filter:", len(clean))

## 3. Duplicates και coordinate ranges

In [ ]:
print(clean[clean["observation_id"].duplicated(keep=False)])
clean = clean.drop_duplicates(subset="observation_id", keep="first")
valid_coordinates = clean["longitude"].between(-180, 180) & clean["latitude"].between(-90, 90)
clean = clean[valid_coordinates].copy()
print("Μετά duplicates και coordinates:", len(clean))

## 4. Geometry και οπτικός έλεγχος

In [ ]:
gdf = gpd.GeoDataFrame(clean, geometry=gpd.points_from_xy(clean["longitude"], clean["latitude"]), crs="EPSG:4326")
print(gdf.geometry.geom_type.value_counts())
print("Bounds:", gdf.total_bounds)
gdf.plot(figsize=(7, 6), color="darkgreen")

## 5. GeoPython QA

In [ ]:
assert len(gdf) > 0
assert gdf.crs.to_string() == "EPSG:4326"
assert gdf.geometry.geom_type.eq("Point").all()
assert gdf.geometry.notna().all()
assert gdf.drop(columns="geometry").isna().sum().sum() == 0
assert gdf.geometry.is_empty.sum() == 0
assert gdf.geometry.is_valid.all()
assert gdf["observation_id"].is_unique
assert gdf["longitude"].between(-180, 180).all()
assert gdf["latitude"].between(-90, 90).all()
print("Όλα τα QA checks πέρασαν.")

## Mini Challenge — Ενδεικτική λύση

In [ ]:
selected = gdf[(gdf["status"] == "active") & (gdf["elevation_m"] > 80)].copy()
assert len(selected) > 0
assert (selected["elevation_m"] > 80).all()
selected.plot(figsize=(7, 6), color="orange")

## Τελικός έλεγχος

Το καθαρό dataset έχει μοναδικά IDs, valid Points, λογικά bounds και CRS EPSG:4326.